In [1]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession

load_dotenv(override=True)

True

In [2]:
agent = Agent(
    name="Jokester",
    instructions="You are a joke teller",
    model="gpt-5.4-mini",
)

In [3]:
result = await Runner.run(agent, "Tell a joke about autonomous AI agents")

In [4]:
print(result.final_output)

Why did the autonomous AI agent start a band?

Because it heard “take initiative” and immediately formed a self-managing startup.


In [5]:
result.to_input_list()

[{'content': 'Tell a joke about autonomous AI agents', 'role': 'user'},
 {'id': 'msg_002f4650efb27ebb006ae3cbb433f487d2b2c0cd095fbcfaf5',
  'content': [{'annotations': [],
    'text': 'Why did the autonomous AI agent start a band?\n\nBecause it heard “take initiative” and immediately formed a self-managing startup.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

## TRACING

In [6]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about autonomous AI agents")

print(result.final_output)

Autonomous AI agents are just interns who never sleep, never ask for PTO, and still somehow need “one tiny clarification” before doing anything.


## streaming

In [7]:
result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI agents.")

async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Sure — here are 5:

1. My AI agent said it could handle anything, so I asked it to organize my life.  
   Now it’s just politely renaming my chaos folders.

2. I built an AI agent to save me time.  
   It worked so well that now I spend all my time supervising it.

3. An AI agent walked into a bar.  
   It immediately asked for the optimal route out.

4. I told my AI agent to be creative.  
   It invented a new way to misunderstand me.

5. Why did the AI agent get promoted?  
   Because it had excellent execution… and no idea what it was actually doing.

## @function_tool

In [14]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

print("user set:", bool(pushover_user))
print("token set:", bool(pushover_token))

user set: True
token set: True


In [15]:
@function_tool
def push_tool(message: str) -> str:
    """Send the given message to the user as a push notification"""
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    status_code = requests.post(pushover_url, data=payload, timeout=10).status_code
    return f"Push sent with API status code {status_code}"

In [16]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000024C9F7FBAD0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [17]:
push_tool.description

'Send the given message to the user as a push notification'

In [18]:
notifier = Agent(
    name="Notifier",
    model="gpt-5.4-mini",
    instructions="You notify the user upon request",
    tools=[push_tool],
)

In [19]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)

Done.


In [20]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [21]:
response = await Runner.run(agent, "Hi there. My name is Siva.")
print(response.final_output)

Hi Siva — nice to meet you! How can I help?


In [22]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don’t know your name unless you tell me.


In [23]:
response = await Runner.run(agent, "Hi there. My name is Siva.")
print(response.final_output)

Hi Siva — nice to meet you! How can I help today?


In [24]:
response.to_input_list()

[{'content': 'Hi there. My name is Siva.', 'role': 'user'},
 {'id': 'msg_034252d040dd7a0e006ae3cf6d37d487d2ba1e54d57bdeb8f1',
  'content': [{'annotations': [],
    'text': 'Hi Siva — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [25]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Siva.', 'role': 'user'},
 {'id': 'msg_034252d040dd7a0e006ae3cf6d37d487d2ba1e54d57bdeb8f1',
  'content': [{'annotations': [],
    'text': 'Hi Siva — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name?"}]

In [26]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Siva.
